In [ ]:
from google.colab import drive
# Mount Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
!pip install peft==0.15.2
!pip install transformers==4.51.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 88.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 87.5 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.15.2
    Uninstalling tokenizers-0.15.2:
      Successfully uninstalled tokenizers-0.15.2
  Attempting uninstall: transformers
    Found existing installation: transformers 4.37.2
    Uninstalling transformers-4.37.2:
      Successfully uninstalled transformers-4.37.2


In [3]:
!pip install langfuse

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.0/275.0 kB 6.3 MB/s eta 0:00:00


# 모델 파인튜닝

In [ ]:
import os
import torch
import numpy as np
from datasets import load_dataset
from sklearn.metrics import accuracy_score
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftConfig,
    PeftModel
)

# ─── 설정 상수 ───────────────────────────────────────────────────────────────
BASE_MODEL_DIR   = "/content/drive/MyDrive/team_project/10-team-matching-quiz-ai/training/model/Qwen_Qwen3-4B-Base"
TRAIN_FILE       = "/content/drive/MyDrive/team_project/10-team-matching-quiz-ai/dataset/processed/train.jsonl"
VAL_FILE         = "/content/drive/MyDrive/team_project/10-team-matching-quiz-ai/dataset/processed/val.jsonl"
OUTPUT_DIR       = "/content/drive/MyDrive/team_project/10-team-matching-quiz-ai/training/model/model_output/qwen3-qlora-output"
MERGED_DIR       = "/content/drive/MyDrive/team_project/10-team-matching-quiz-ai/training/model/model_output/qwen3-qlora-merged"
MAX_LENGTH       = 1024
RESUME_TRAINING  = True
BATCH_SIZE       = 4
ACCUM_STEPS      = 8
NUM_EPOCHS       = 3
LEARNING_RATE    = 2e-4

# ─── 1. 토크나이저 로드 ───────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_DIR,
    trust_remote_code=True
)
tokenizer.pad_token = tokenizer.eos_token

# ─── 2. QLoRA 양자화 모델 로드 ────────────────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_DIR,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
model = prepare_model_for_kbit_training(model)

# ─── 3. LoRA 설정 및 모델 래핑 ─────────────────────────────────────────────────
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

# ─── 4. 데이터셋 준비 ─────────────────────────────────────────────────────────
dataset = load_dataset("json", data_files={"train": TRAIN_FILE, "eval": VAL_FILE})

def tokenize_fn(examples):
    enc = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH
    )
    enc["labels"] = enc["input_ids"].copy()
    return enc

train_ds = dataset["train"].map(
    tokenize_fn,
    batched=True,
    remove_columns=["text"]
)
eval_ds = dataset["eval"].map(
    tokenize_fn,
    batched=True,
    remove_columns=["text"]
)

# ─── 5. 평가 메트릭 정의 ───────────────────────────────────────────────────────
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    mask = labels != -100
    labels_f = labels[mask]
    preds_f = preds[mask]
    acc = accuracy_score(labels_f, preds_f)

    # Perplexity 계산
    flat_logits = torch.from_numpy(logits).view(-1, logits.shape[-1])
    flat_labels = torch.from_numpy(labels).view(-1)
    active = flat_labels != -100
    active_logits = flat_logits[active]
    active_labels = flat_labels[active]
    loss_fct = torch.nn.CrossEntropyLoss()
    loss = loss_fct(active_logits, active_labels)
    ppl = torch.exp(loss).item()

    return {"accuracy": acc, "perplexity": ppl}

# ─── 6. 학습 인자 설정 ─────────────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=ACCUM_STEPS,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    logging_steps=20,
    save_strategy="epoch",
    eval_strategy="epoch",
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="perplexity",
    greater_is_better=False,
    fp16=True,
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    warmup_steps=100,
)

torch.cuda.empty_cache()

# ─── 7. Trainer 초기화 및 학습 시작 ─────────────────────────────────────────────
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    compute_metrics=compute_metrics
)

checkpoint = None
if RESUME_TRAINING and os.path.isdir(OUTPUT_DIR):
    ckpts = [d for d in os.listdir(OUTPUT_DIR) if d.startswith("checkpoint-")]
    if ckpts:
        latest = sorted(ckpts, key=lambda x: int(x.split("-")[1]))[-1]
        checkpoint = os.path.join(OUTPUT_DIR, latest)
        print(f"✅ Resume from {checkpoint}")

trainer.train(resume_from_checkpoint=checkpoint)
model.save_pretrained(OUTPUT_DIR)
torch.cuda.empty_cache()

# ─── 8. LoRA 병합 및 저장 ─────────────────────────────────────────────────────
peft_conf = PeftConfig.from_pretrained(OUTPUT_DIR)
base_model = AutoModelForCausalLM.from_pretrained(
    peft_conf.base_model_name_or_path,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
merged = PeftModel.from_pretrained(base_model, OUTPUT_DIR, torch_dtype=torch.float16)
merged = merged.merge_and_unload()
merged.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)

print("QLoRA 파인튜닝 완료!")



Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Map:   0%|          | 0/2198 [00:00<?, ? examples/s]

Map:   0%|          | 0/550 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: taeyoung-kong (taeyoung-kong-discord) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss
